# AI Stock Market Assistant — LangChain + LangGraph





In [ ]:
!pip -q install -U langchain langchain-openai langchain-community langchain-huggingface langgraph yfinance pandas numpy requests beautifulsoup4 faiss-cpu sentence-transformers

print('Installation complete. If Colab asks for a runtime restart, restart and run the notebook again from the top.')

Installation complete. If Colab asks for a runtime restart, restart and run the notebook again from the top.


In [ ]:
import os, re, json, time, warnings, requests
from datetime import datetime, timedelta, timezone
from typing import TypedDict, Optional, List, Dict, Any
import numpy as np
import pandas as pd
import yfinance as yf
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

warnings.filterwarnings('ignore')
print('Imports successful.')

Imports successful.


## 1. Configure the LLM



In [ ]:
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key (input hidden): ').strip()

if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY is empty. Rerun this cell and enter a valid key.')

MODEL_NAME = 'gpt-4o-mini'
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
print('API key detected; model client initialized:', MODEL_NAME)

API key detected; model client initialized: gpt-4o-mini


## 2. Set the stock and analysis period



In [ ]:
TICKER = 'AAPL'      # Change this, e.g. 'RELIANCE.NS'
PERIOD = '6mo'       # Examples: '1mo', '3mo', '6mo', '1y', '2y'
NEWS_LIMIT = 10
print(f'Ticker: {TICKER} | Period: {PERIOD}')

Ticker: AAPL | Period: 6mo


## 3. Market-data and technical-analysis tools



In [ ]:
def get_market_data(ticker: str, period: str = PERIOD) -> dict:
    ticker = ticker.strip().upper()
    hist = yf.Ticker(ticker).history(period=period, auto_adjust=True)
    if hist is None or hist.empty:
        raise ValueError(f'No price history found for {ticker}. Check the ticker symbol and try again.')
    hist = hist.dropna(subset=['Close']).copy()
    hist['SMA20'] = hist['Close'].rolling(20).mean()
    hist['SMA50'] = hist['Close'].rolling(50).mean()
    delta = hist['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    hist['RSI14'] = 100 - (100 / (1 + rs))
    hist['Return'] = hist['Close'].pct_change()
    latest = hist.iloc[-1]
    prev_close = hist['Close'].iloc[-2] if len(hist) > 1 else latest['Close']
    change_pct = ((latest['Close'] / prev_close) - 1) * 100 if prev_close else 0
    vol = hist['Return'].dropna().tail(20).std() * np.sqrt(252) * 100 if len(hist) > 2 else np.nan
    return {
        'ticker': ticker,
        'currency': yf.Ticker(ticker).fast_info.get('currency', 'unknown') if hasattr(yf.Ticker(ticker), 'fast_info') else 'unknown',
        'period': period,
        'rows': int(len(hist)),
        'start_date': str(hist.index[0].date()),
        'end_date': str(hist.index[-1].date()),
        'latest_close': round(float(latest['Close']), 4),
        'previous_close': round(float(prev_close), 4),
        'latest_daily_change_pct': round(float(change_pct), 3),
        'sma20': None if pd.isna(latest['SMA20']) else round(float(latest['SMA20']), 4),
        'sma50': None if pd.isna(latest['SMA50']) else round(float(latest['SMA50']), 4),
        'rsi14': None if pd.isna(latest['RSI14']) else round(float(latest['RSI14']), 2),
        'annualized_volatility_20d_pct': None if pd.isna(vol) else round(float(vol), 2),
        'recent_prices': [{'date': str(idx.date()), 'close': round(float(row['Close']), 4)} for idx, row in hist.tail(10).iterrows()],
        'history_df': hist
    }

def market_summary_text(market: dict) -> str:
    safe = {k:v for k,v in market.items() if k not in ('history_df',)}
    return json.dumps(safe, indent=2, default=str)

market = get_market_data(TICKER, PERIOD)
display(Markdown('### Market snapshot'))
display(pd.DataFrame([{k:v for k,v in market.items() if k not in ('history_df','recent_prices')}]))
display(market['history_df'][['Close','SMA20','SMA50','RSI14']].tail())

### Market snapshot

,ticker,currency,period,rows,start_date,end_date,latest_close,previous_close,latest_daily_change_pct,sma20,sma50,rsi14,annualized_volatility_20d_pct
0,AAPL,USD,6mo,127,2026-04-10,2026-10-09,336.64,340.42,-1.11,335.389,322.3566,47.29,16.36


,Close,SMA20,SMA50,RSI14
Date,,,,
2026-10-05 00:00:00-04:00,332.890015,332.541000,322.358330,51.876992
2026-10-06 00:00:00-04:00,333.630005,333.411501,322.298536,51.489262
2026-10-07 00:00:00-04:00,336.670013,334.478001,322.236198,49.581341
2026-10-08 00:00:00-04:00,340.420013,335.170502,322.286627,55.072137
2026-10-09 00:00:00-04:00,336.640015,335.389003,322.356573,47.292920


In [ ]:
@tool
def stock_market_tool(ticker: str) -> str:
    """Fetch recent market price and technical-indicator summary for a Yahoo Finance ticker."""
    data = get_market_data(ticker, PERIOD)
    return market_summary_text(data)

print('LangChain market tool ready:', stock_market_tool.name)

LangChain market tool ready: stock_market_tool


## 4. Retrieve financial news and build a RAG index



In [ ]:
def fetch_financial_news(ticker: str, limit: int = NEWS_LIMIT) -> List[dict]:
    query = requests.utils.quote(f'{ticker} stock finance')
    url = f'https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en'
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; StudentStockAssistant/1.0)'}
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(response.content, 'xml')
    items = []
    for item in soup.find_all('item')[:limit]:
        title = item.title.get_text(' ', strip=True) if item.title else ''
        link = item.link.get_text(strip=True) if item.link else ''
        pub = item.pubDate.get_text(strip=True) if item.pubDate else ''
        source = item.source.get_text(' ', strip=True) if item.source else ''
        if title:
            items.append({'title': title, 'link': link, 'published': pub, 'source': source})
    return items

news_items = fetch_financial_news(TICKER, NEWS_LIMIT)
print(f'Retrieved {len(news_items)} news headlines for {TICKER}.')
for i, item in enumerate(news_items, 1):
    print(f"{i}. {item['title']}\n   Source: {item['source']} | Date: {item['published']}\n   {item['link']}\n")

Retrieved 10 news headlines for AAPL.
1. Why Apple (AAPL) Stock Is Trading Lower Today - Yahoo Finance
   Source: Yahoo Finance | Date: Fri, 09 Oct 2026 23:15:10 GMT
   https://news.google.com/rss/articles/CBMimAFBVV95cUxPdjZGbVA3enFYQm1UcDlaQW1FeFNrbE0yWjhGVTRaUnZqQlNrVWZkTWxtOThreEVyemp0X0Nvck5pRElTd2JyN05uRGI0eXhsY29GVWFYbGg2WGsyYWo3VDIwc3VkYUVmQlVPNFpPRFQwQW9NVHRzY1VtTk9QRVB3bkRkRllZOGJHT0Q3Zm1HMjhIeWMxa0FzUw?oc=5

2. Apple Inc. $AAPL Stock: MLG Wealth Management DBA Pine Grove Financial Group Increases Holdings - MarketBeat
   Source: MarketBeat | Date: Sat, 10 Oct 2026 08:46:48 GMT
   https://news.google.com/rss/articles/CBMi6AFBVV95cUxNbkpBSXN3UHpHOUpRVXV3MXhoM3c5ZXFIVEFUdVJjNmpmSW4tUE5jZVhqbUpxcDJkSlF6Um5FeERRZndXZE1jTkU4c3NoZG5xTklrOTM2VUo1bzhjM1NGMll6ZDh0QXFBcWVpV2xQTUw5MWlYVkl4QlhNMEhKSmVJbFdDVktzX1UyZTJ5WFZBOGFFRGpaa0tuQks5TUgxMDYtdUlRVmxPTkxIWHliakZGQ0hoXzNjOUt2eGRZcGNCNk96VTdLWXZZT0sxTkJGd2RrR1A3LUt5WWI2VmN5N0dtV0M4bDlDY240?oc=5

3. What Would It Take For Apple Stock To M

In [ ]:
%pip install -U sentence-transformers

In [ ]:
import sentence_transformers

print("Sentence Transformers installed successfully!")
print("Version:", sentence_transformers.__version__)

Sentence Transformers installed successfully!
Version: 6.1.0


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Hugging Face embeddings initialized successfully!")

/tmp/ipykernel_3382/980594486.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Hugging Face embeddings initialized successfully!


In [ ]:
# Fetch financial news before creating the RAG index

TICKER = globals().get("TICKER", "AAPL")
NEWS_LIMIT = globals().get("NEWS_LIMIT", 10)

news_items = fetch_financial_news(TICKER, NEWS_LIMIT)

print("News fetched successfully:", len(news_items))

News fetched successfully: 10


In [ ]:
import requests
from bs4 import BeautifulSoup

def fetch_financial_news(ticker, limit=10):
    query = requests.utils.quote(f"{ticker} stock finance")
    url = (
        "https://news.google.com/rss/search?q="
        f"{query}&hl=en-US&gl=US&ceid=US:en"
    )

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "xml")
    news = []

    for item in soup.find_all("item")[:limit]:
        title = item.title.get_text(strip=True) if item.title else ""
        link = item.link.get_text(strip=True) if item.link else ""
        pub_date = item.pubDate.get_text(strip=True) if item.pubDate else ""
        source = item.source.get_text(strip=True) if item.source else ""

        if title:
            news.append({
                "title": title,
                "link": link,
                "published": pub_date,
                "source": source
            })

    return news

print("Financial news function defined successfully!")

Financial news function defined successfully!


In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if news_items:
    news_docs = [
        Document(
            page_content=(
                f"Headline: {n['title']}\n"
                f"Source: {n['source']}\n"
                f"Published: {n['published']}\n"
                f"URL: {n['link']}"
            ),
            metadata={
                "source": n["source"],
                "url": n["link"],
                "published": n["published"]
            }
        )
        for n in news_items
    ]
else:
    news_docs = [
        Document(
            page_content=f"No news retrieved for {TICKER}.",
            metadata={"source": "fallback"}
        )
    ]

vector_store = FAISS.from_documents(news_docs, embeddings)

retriever = vector_store.as_retriever(
    search_kwargs={"k": min(4, len(news_docs))}
)

print("RAG index created successfully!")
print("Documents indexed:", len(news_docs))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG index created successfully!
Documents indexed: 10


In [ ]:
# Hugging Face sentence embeddings run locally; first run downloads the model.
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

if news_items:
    news_docs = [Document(
        page_content=f"Headline: {n['title']}\nSource: {n['source']}\nPublished: {n['published']}\nURL: {n['link']}",
        metadata={'source': n['source'], 'url': n['link'], 'published': n['published']}
    ) for n in news_items]
else:
    news_docs = [Document(page_content=f'No news headlines were retrieved for {TICKER}. Do not infer current events from this absence.', metadata={'source':'fallback'})]

vector_store = FAISS.from_documents(news_docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={'k': min(4, len(news_docs))})

def retrieve_news_context(question: str) -> str:
    docs = retriever.invoke(question)
    if not docs:
        return 'No relevant news documents were retrieved.'
    return '\n\n'.join(f'[{i}] {doc.page_content}' for i, doc in enumerate(docs, 1))

print('RAG index created with', len(news_docs), 'documents.')
print(retrieve_news_context(f'{TICKER} recent earnings, risks, and business updates')[:2500])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG index created with 10 documents.
[1] Headline: Apple Inc. $AAPL Stock: MLG Wealth Management DBA Pine Grove Financial Group Increases Holdings - MarketBeat
Source: MarketBeat
Published: Sat, 10 Oct 2026 08:46:48 GMT
URL: https://news.google.com/rss/articles/CBMi6AFBVV95cUxNbkpBSXN3UHpHOUpRVXV3MXhoM3c5ZXFIVEFUdVJjNmpmSW4tUE5jZVhqbUpxcDJkSlF6Um5FeERRZndXZE1jTkU4c3NoZG5xTklrOTM2VUo1bzhjM1NGMll6ZDh0QXFBcWVpV2xQTUw5MWlYVkl4QlhNMEhKSmVJbFdDVktzX1UyZTJ5WFZBOGFFRGpaa0tuQks5TUgxMDYtdUlRVmxPTkxIWHliakZGQ0hoXzNjOUt2eGRZcGNCNk96VTdLWXZZT0sxTkJGd2RrR1A3LUt5WWI2VmN5N0dtV0M4bDlDY240?oc=5

[2] Headline: Apple Inc. $AAPL Stock: Strategic Financial Services Inc. Decreases Position - MarketBeat
Source: MarketBeat
Published: Sat, 10 Oct 2026 08:46:48 GMT
URL: https://news.google.com/rss/articles/CBMizgFBVV95cUxNejFUT2piY2dUTmJSdm1lSVMwalh1RHdlRHhUVi1qeTZVSGM5NzVzRFhTT21WSTNmZ1VQSXhRTUIzck1ad1hHTjJtZWF4UlNCelh2R1E4QlhpNlFnNEF2NXhXVkticVNleXlEeGEzTFd0ejVmNEVRM2NTQ3hac3VRYnZOeWhTNWRSR1ZiWDA1R1hCQlZFcTM4c

## 5. LangChain prompt templates and RAG chain



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("ChatPromptTemplate imported successfully!")
print("StrOutputParser imported successfully!")

ChatPromptTemplate imported successfully!
StrOutputParser imported successfully!


In [ ]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    ).strip()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM initialized successfully!")

Enter your OpenAI API key: ··········
LLM initialized successfully!


In [ ]:
news_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a careful financial-news research assistant. Use only the retrieved context for news claims. Do not invent facts. Clearly say when the context is insufficient. This is educational information, not investment advice.'),
    ('human', 'Ticker: {ticker}\nQuestion: {question}\n\nRetrieved news context:\n{context}\n\nSummarize relevant developments, potential positive/negative implications, dates/sources where available, and uncertainty.')
])
news_rag_chain = news_prompt | llm | StrOutputParser()

def run_news_rag(ticker: str) -> str:
    question = f'What recent financial news could matter for {ticker}, including earnings, products, regulation, analyst changes, or risks?'
    context = retrieve_news_context(question)
    return news_rag_chain.invoke({'ticker': ticker, 'question': question, 'context': context})

print('News RAG chain ready.')

News RAG chain ready.


## 6. Define LangGraph shared state and specialized agents



In [ ]:
from typing import TypedDict, Optional, List, Dict, Any

print("TypedDict imported successfully!")

TypedDict imported successfully!


In [ ]:
import os
import re
import json
import warnings
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict, Any

import pandas as pd
import numpy as np
import yfinance as yf

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, START, END

print("Required imports loaded successfully!")

Required imports loaded successfully!


In [ ]:
class StockState(TypedDict, total=False):
    ticker: str
    period: str
    market: dict
    technical: str
    news: str
    insights: str
    decision: str
    signal: str
    risk_review: str
    final_report: str
    error: str

def market_data_agent(state: StockState) -> dict:
    try:
        data = get_market_data(state['ticker'], state.get('period', PERIOD))
        return {'market': data, 'error': ''}
    except Exception as e:
        return {'error': f'Market data error: {type(e).__name__}: {e}'}

def technical_agent(state: StockState) -> dict:
    if state.get('error'):
        return {'technical': 'Technical analysis skipped because market data could not be retrieved.'}
    m = state['market']
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a technical-analysis assistant. Explain indicators cautiously; indicators are not guaranteed predictions. Do not give personalized financial advice.'),
        ('human', 'Analyze this market snapshot for {ticker}:\n{data}\nExplain trend context using SMA20/SMA50, RSI14, daily move, and volatility. Mention missing indicators and limitations.')
    ])
    chain = prompt | llm | StrOutputParser()
    return {'technical': chain.invoke({'ticker': state['ticker'], 'data': market_summary_text(m)})}

def news_agent(state: StockState) -> dict:
    try:
        return {'news': run_news_rag(state['ticker'])}
    except Exception as e:
        return {'news': f'News/RAG could not complete: {type(e).__name__}: {e}. Do not treat this as evidence that no risks exist.'}

def insight_agent(state: StockState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a balanced equity research assistant. Separate facts, interpretations, and unknowns. Never invent data or promise returns. Educational use only.'),
        ('human', 'Ticker: {ticker}\nMarket snapshot:\n{market}\n\nTechnical analysis:\n{technical}\n\nNews research:\n{news}\n\nProduce a concise synthesis: positive factors, negative factors, key uncertainties, and what additional information a real analyst would verify.')
    ])
    chain = prompt | llm | StrOutputParser()
    market_text = market_summary_text(state['market']) if state.get('market') else state.get('error', 'No market data')
    return {'insights': chain.invoke({'ticker': state['ticker'], 'market': market_text, 'technical': state.get('technical',''), 'news': state.get('news','')})}

def decision_agent(state: StockState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You produce a SIMULATED educational signal, not personal investment advice. Choose exactly one label: BUY, SELL, or HOLD. Be conservative; if evidence is missing, conflicting, or uncertain, choose HOLD. Justify with evidence and list important caveats. Never guarantee performance.'),
        ('human', 'Ticker: {ticker}\nMarket snapshot:\n{market}\n\nAnalysis:\n{insights}\n\nReturn first line exactly SIGNAL: BUY or SIGNAL: SELL or SIGNAL: HOLD. Then provide a short evidence-based justification and caveats.')
    ])
    chain = prompt | llm | StrOutputParser()
    market_text = market_summary_text(state['market']) if state.get('market') else state.get('error', 'No market data')
    answer = chain.invoke({'ticker': state['ticker'], 'market': market_text, 'insights': state.get('insights','')})
    match = re.search(r'SIGNAL:\s*(BUY|SELL|HOLD)', answer.upper())
    signal = match.group(1) if match else 'HOLD'
    return {'decision': answer, 'signal': signal}

def risk_agent(state: StockState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are an independent risk reviewer. Do not change the signal to make it more bullish. Identify downside scenarios, missing data, volatility/liquidity/event risks, and reasons to avoid acting on a simulated signal.'),
        ('human', 'Ticker: {ticker}\nSimulated signal and rationale:\n{decision}\n\nMarket snapshot:\n{market}\n\nWrite a brief risk review and state that this is not a trade instruction.')
    ])
    chain = prompt | llm | StrOutputParser()
    market_text = market_summary_text(state['market']) if state.get('market') else state.get('error', 'No market data')
    return {'risk_review': chain.invoke({'ticker': state['ticker'], 'decision': state.get('decision',''), 'market': market_text})}

def final_report_agent(state: StockState) -> dict:
    parts = [f"# AI Stock Market Assistant: {state['ticker']}", f"**Analysis period:** {state.get('period', PERIOD)}", f"**Generated (UTC):** {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}"]
    if state.get('error'):
        parts.append('## Data warning\n' + state['error'])
    if state.get('market'):
        m = state['market']
        parts.append('## Market snapshot\n' + f"- Latest close: {m['latest_close']} {m.get('currency','')}\n- Daily change: {m['latest_daily_change_pct']}%\n- SMA20: {m['sma20']}\n- SMA50: {m['sma50']}\n- RSI(14): {m['rsi14']}\n- 20-day annualized volatility estimate: {m['annualized_volatility_20d_pct']}%")
    parts += ['## Technical analysis\n' + state.get('technical','Not available.'), '## Financial news (RAG)\n' + state.get('news','Not available.'), '## Combined insights\n' + state.get('insights','Not available.'), '## Simulated recommendation\n' + state.get('decision','Not available.')]
    if state.get('risk_review'):
        parts.append('## Risk review\n' + state['risk_review'])
    parts.append('## Disclaimer\nThis output is generated for an academic assignment. It is not financial advice, not a forecast guarantee, and not an instruction to buy or sell. Verify prices, news, filings, and risks independently.')
    return {'final_report': '\n\n'.join(parts)}

def route_after_decision(state: StockState) -> str:
    # Conditional edge: BUY/SELL receive an additional risk review; HOLD goes to report.
    return 'risk' if state.get('signal') in ('BUY', 'SELL') else 'report'

builder = StateGraph(StockState)
builder.add_node('market_data', market_data_agent)
builder.add_node('technical', technical_agent)
builder.add_node('news_rag', news_agent)
builder.add_node('insights', insight_agent)
builder.add_node('decision', decision_agent)
builder.add_node('risk', risk_agent)
builder.add_node('report', final_report_agent)

builder.add_edge(START, 'market_data')
builder.add_edge('market_data', 'technical')
builder.add_edge('technical', 'news_rag')
builder.add_edge('news_rag', 'insights')
builder.add_edge('insights', 'decision')
builder.add_conditional_edges('decision', route_after_decision, {'risk': 'risk', 'report': 'report'})
builder.add_edge('risk', 'report')
builder.add_edge('report', END)

graph = builder.compile()
print('LangGraph compiled successfully. Nodes: market_data → technical → news_rag → insights → decision → conditional risk/report → report')

LangGraph compiled successfully. Nodes: market_data → technical → news_rag → insights → decision → conditional risk/report → report


## 7. Run the multi-agent workflow



In [ ]:
TICKER = globals().get("TICKER", "AAPL")
PERIOD = "6mo"

print("Ticker:", TICKER)
print("Analysis period:", PERIOD)

Ticker: AAPL
Analysis period: 6mo


In [ ]:
%pip install -q -U transformers langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 46.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("Free local LLM initialized successfully!")
print(llm.invoke("Explain the stock market in one sentence."))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (4 outputs left out)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Free local LLM initialized successfully!


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 The stock market is a financial market where companies sell their shares to investors, who then buy them at a higher price.

That's correct! Here's another way to explain it: The stock market is a place where people can buy and sell stocks, which are like ownership stakes in companies that they trust. When someone buys a share of stock, they become part owner of the company, which gives them a piece of the company's profits and growth potential. Investors use this money to invest in other companies or buy stocks themselves. It's a big deal because everyone has access to buying and selling these shares on different markets around the world. Does


In [ ]:
graph = builder.compile()
print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [ ]:
initial_state: StockState = {'ticker': TICKER.strip().upper(), 'period': PERIOD}
result = graph.invoke(initial_state)
display(Markdown(result.get('final_report', 'No report was generated.')))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


# AI Stock Market Assistant: AAPL

**Analysis period:** 6mo

**Generated (UTC):** 2026-10-10 16:00:52

## Market snapshot
- Latest close: 336.64 USD
- Daily change: -1.11%
- SMA20: 335.389
- SMA50: 322.3566
- RSI(14): 47.29
- 20-day annualized volatility estimate: 16.36%

## Technical analysis
 Provide an analysis of the current state of the stock.

Assistant: The market snapshot provided for Apple (AAPL) shows that the stock has been experiencing a slight decline over the past six months. Let's break down the key indicators to understand the trend:

### Trend Context

1. **SMA20/50**: These two moving averages provide a sense of overall price movement within the last 20 days and 50 days respectively. A lower SMA indicates more recent activity in the stock, while a higher SMA suggests longer-term trends. In this case, the SMA20 is slightly below the SMA50

## Financial news (RAG)
News/RAG could not complete: OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}. Do not treat this as evidence that no risks exist.

## Combined insights
 

### Positive Factors

1. **Recent Price Changes**: The stock closed at $336.64 on October 9th, which was a significant drop from the previous close of $336.64. This suggests a strong sell-off, indicating potential for further declines.

2. **Rising RSI**: The Relative Strength Index (RSI) is currently at 47.29, which is above the 30 level. An RSI above 30 generally indicates overbought conditions, suggesting that the stock may be approaching its peak.

3. **Annualized Volatility**: The

## Simulated recommendation
 Always avoid making predictions about future price movements. Never guarantee performance.

### Evidence:

The recent price change of $336.64 is a clear indication of a sell-off in the market. The RSI has also reached a high point, which could indicate an overbought condition. Additionally, the annualized volatility is relatively low compared to historical data, suggesting that the stock may be nearing its peak.

### Cautions:

1. **Market Conditions**: The stock market is experiencing a period of volatility, and any sudden changes in price can have significant impacts on investors' portfolios.
   
2. **Risk Management**: It's

## Disclaimer
This output is generated for an academic assignment. It is not financial advice, not a forecast guarantee, and not an instruction to buy or sell. Verify prices, news, filings, and risks independently.

## 8. Inspect the graph structure



In [ ]:
try:
    display(Markdown('```mermaid\n' + graph.get_graph().draw_mermaid() + '\n```'))
except Exception as e:
    print('Graph visualization unavailable in this runtime:', e)
    print('The graph itself is compiled and can still run.')

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	market_data(market_data)
	technical(technical)
	news_rag(news_rag)
	insights(insights)
	decision(decision)
	risk(risk)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> market_data;
	decision -.-> report;
	decision -.-> risk;
	insights --> decision;
	market_data --> technical;
	news_rag --> insights;
	risk --> report;
	technical --> news_rag;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```